# Определение стоимости автомобилей

Сервис по продаже автомобилей с пробегом «Не бит, не крашен» разрабатывает приложение для привлечения новых клиентов. В нём можно быстро узнать рыночную стоимость своего автомобиля. В вашем распоряжении исторические данные: технические характеристики, комплектации и цены автомобилей. Вам нужно построить модель для определения стоимости. 

Заказчику важны:

- качество предсказания;
- скорость предсказания;
- время обучения.

## Подготовка данных

На данном шаге:
- Импортируем библиотеки. 
- Откроем файл и изучим его.
- Проведем предобработку данных.

Для начала импортируем все необходимые библиотеки для работы:

In [1]:
import pandas as pd
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OrdinalEncoder 
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
pd.set_option('display.max_rows', None)
import warnings
warnings.filterwarnings('ignore')
import time

Откроем файл с данными.

In [2]:
df = pd.read_csv('/datasets/autos.csv')

Изучим первые 5 строк датасета.

In [3]:
df.head()

,DateCrawled,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,RegistrationMonth,FuelType,Brand,NotRepaired,DateCreated,NumberOfPictures,PostalCode,LastSeen
0,2016-03-24 11:52:17,480,NaN,1993,manual,0,golf,150000,0,petrol,volkswagen,NaN,2016-03-24 00:00:00,0,70435,2016-04-07 03:16:57
1,2016-03-24 10:58:45,18300,coupe,2011,manual,190,NaN,125000,5,gasoline,audi,yes,2016-03-24 00:00:00,0,66954,2016-04-07 01:46:50
2,2016-03-14 12:52:21,9800,suv,2004,auto,163,grand,125000,8,gasoline,jeep,NaN,2016-03-14 00:00:00,0,90480,2016-04-05 12:47:46
3,2016-03-17 16:54:04,1500,small,2001,manual,75,golf,150000,6,petrol,volkswagen,no,2016-03-17 00:00:00,0,91074,2016-03-17 17:40:17
4,2016-03-31 17:25:20,3600,small,2008,manual,69,fabia,90000,7,gasoline,skoda,no,2016-03-31 00:00:00,0,60437,2016-04-06 10:17:21


Изучим общую информацию о датафреймах.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Kilometer          354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

# Описание данных

Признаки 

- DateCrawled — дата скачивания анкеты из базы
- VehicleType — тип автомобильного кузова
- RegistrationYear — год регистрации автомобиля
- Gearbox — тип коробки передач
- Power — мощность (л. с.)
- Model — модель автомобиля
- Kilometer — пробег (км)
- RegistrationMonth — месяц регистрации автомобиля
- FuelType — тип топлива
- Brand — марка автомобиля
- NotRepaired — была машина в ремонте или нет
- DateCreated — дата создания анкеты
- NumberOfPictures — количество фотографий автомобиля
- PostalCode — почтовый индекс владельца анкеты (пользователя)
- LastSeen — дата последней активности пользователя

Целевой признак
- Price — цена (евро)

# Предобработка данных

Удалим ненужные столбцы с данными, которые не влияют на качество предсказания.

In [5]:
df = df.drop(["DateCrawled","DateCreated", 'RegistrationMonth',"LastSeen","NumberOfPictures","PostalCode"],axis = 1)

Проверим данные на наличие дубликатов.

In [6]:
df.duplicated().sum()

45040

Удалим дубликаты из датасета.

In [7]:
df.drop_duplicates(inplace=True)

Изучим численные показатели датафрейма.

In [8]:
df.describe()

,Price,RegistrationYear,Power,Kilometer
count,309329.000000,309329.000000,309329.000000,309329.000000
mean,4486.937196,2004.360105,110.976908,127217.735809
std,4564.852796,92.541399,200.969473,38532.941010
min,0.000000,1000.000000,0.000000,5000.000000
25%,1100.000000,1999.000000,69.000000,125000.000000
50%,2800.000000,2003.000000,105.000000,150000.000000
75%,6500.000000,2008.000000,143.000000,150000.000000
max,20000.000000,9999.000000,20000.000000,150000.000000


Видим, что в столбцах RegistrationYear, Power присутствуют аномальные значения. Необходимо их удалить. Напишем функцию, которая удалет выбросы методом межквартильного размаха.

In [9]:
def emission(df, column):
    q1 = df[column].quantile(q=0.25)
    q3 = df[column].quantile(q=0.75)
    norm = df.loc[(df[column] < q3 + 1.5*(q3 - q1)) & (df[column] > q1 - 1.5*(q3-q1)), column]
    return norm

emission_list = ['RegistrationYear','Power']
for item in emission_list:
    df[item] = emission(df, item)

Посмортим снова на численные показатели датафрейма.

In [10]:
df.describe()

,Price,RegistrationYear,Power,Kilometer
count,309329.000000,303327.000000,302599.000000,309329.000000
mean,4486.937196,2003.695500,102.665878,127217.735809
std,4564.852796,6.479498,57.149181,38532.941010
min,0.000000,1986.000000,0.000000,5000.000000
25%,1100.000000,1999.000000,68.000000,125000.000000
50%,2800.000000,2003.000000,102.000000,150000.000000
75%,6500.000000,2008.000000,140.000000,150000.000000
max,20000.000000,2019.000000,253.000000,150000.000000


In [11]:
df[df['Power']>=250]

,Price,VehicleType,RegistrationYear,Gearbox,Power,Model,Kilometer,FuelType,Brand,NotRepaired
190,11299,sedan,2006.0,auto,250.0,golf,150000,petrol,volkswagen,no
661,9900,suv,2000.0,auto,250.0,NaN,150000,lpg,sonstige_autos,no
1137,5699,wagon,2004.0,auto,250.0,a6,150000,petrol,audi,NaN
1271,9999,suv,2005.0,auto,250.0,m_klasse,150000,gasoline,mercedes_benz,no
1557,7999,coupe,2004.0,auto,250.0,a3,150000,petrol,audi,no
2152,5000,suv,2002.0,auto,250.0,m_klasse,150000,gasoline,mercedes_benz,no
2388,3330,sedan,1991.0,auto,252.0,other,150000,petrol,mercedes_benz,no
2699,4990,wagon,2003.0,auto,250.0,other,150000,petrol,saab,no
3026,14999,convertible,2000.0,manual,252.0,boxster,125000,petrol,porsche,no
3319,18500,wagon,2013.0,manual,250.0,focus,60000,petrol,ford,no


Аномальных значений больше нет.

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 309329 entries, 0 to 354368
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   Price             309329 non-null  int64  
 1   VehicleType       274770 non-null  object 
 2   RegistrationYear  303327 non-null  float64
 3   Gearbox           292122 non-null  object 
 4   Power             302599 non-null  float64
 5   Model             290968 non-null  object 
 6   Kilometer         309329 non-null  int64  
 7   FuelType          278565 non-null  object 
 8   Brand             309329 non-null  object 
 9   NotRepaired       244771 non-null  object 
dtypes: float64(2), int64(2), object(6)
memory usage: 26.0+ MB


Изучим столбцы с категориальными данными.

In [13]:
cat_features = ['VehicleType','Gearbox','Model','FuelType','Brand','NotRepaired']
for columns in cat_features:
    print(df[columns].value_counts(normalize=True))


sedan          0.284624
small          0.245835
wagon          0.205142
bus            0.094006
convertible    0.066219
coupe          0.053463
suv            0.039218
other          0.011493
Name: VehicleType, dtype: float64
manual    0.796838
auto      0.203162
Name: Gearbox, dtype: float64
golf                  0.081243
other                 0.078820
3er                   0.057395
polo                  0.036066
corsa                 0.033468
astra                 0.032251
passat                0.028556
a4                    0.028415
c_klasse              0.024577
5er                   0.022948
e_klasse              0.020645
a3                    0.018002
focus                 0.017627
fiesta                0.017373
a6                    0.016648
transporter           0.014730
2_reihe               0.014345
twingo                0.013768
fortwo                0.013665
a_klasse              0.013022
vectra                0.012819
mondeo                0.010847
3_reihe               0.

C категориальными данными все в порядке.

Изучим пропускки в датафрейме.

In [14]:
df.isna().sum()

Price                   0
VehicleType         34559
RegistrationYear     6002
Gearbox             17207
Power                6730
Model               18361
Kilometer               0
FuelType            30764
Brand                   0
NotRepaired         64558
dtype: int64

Видим, что есть пропуски во всех столбцах, кроме Price, Brand, Kilometer.

Посмотрим на соотношение значений в столбце NotRepaired.

In [15]:
df['NotRepaired'].value_counts(normalize=True)

no     0.86337
yes    0.13663
Name: NotRepaired, dtype: float64

Видно, что большинство машин не было в ремонте. Значит скорее всего пропущенные значения в столбце NotRepaired означают, что машина не ремонтировалась.

In [16]:
df['NotRepaired'] = df['NotRepaired'].fillna('no')

Остальные пропуски в столбцах c категориальными данными заполним значением 'other'.

In [17]:
df['FuelType'] = df['FuelType'].fillna('other')

In [18]:
df['Gearbox'] = df['Gearbox'].fillna('other')

In [19]:
df['VehicleType'] = df['VehicleType'].fillna('other')

In [20]:
df['Model'] = df['Model'].fillna('other')

Посмотрим на пропуски в столбцах c численными данными. Проанализируем матрицу корреляции.

In [21]:
df.corr()

,Price,RegistrationYear,Power,Kilometer
Price,1.000000,0.458858,0.449806,-0.325592
RegistrationYear,0.458858,1.000000,0.081080,-0.279207
Power,0.449806,0.081080,1.000000,0.100225
Kilometer,-0.325592,-0.279207,0.100225,1.000000


Заполним пропуски в столбце RegistrationYear на медианное значение, сгруппированое по столбцу Price.

In [22]:
df['RegistrationYear'] = df.groupby(['Price'])['RegistrationYear'].apply(lambda x: x.fillna(x.median()))
df['RegistrationYear'].fillna(df['RegistrationYear'].median(), inplace=True)

Заполним пропуски в столбце Power на медианное значение, сгруппированое по столбцу Price.

In [23]:
df['Power'] = df.groupby(['Price'])['Power'].apply(lambda x: x.fillna(x.median()))
df['Power'].fillna(df['Power'].median(), inplace=True)

Посчитаем пропущенные значения снова.

In [24]:
df.isna().sum()

Price               0
VehicleType         0
RegistrationYear    0
Gearbox             0
Power               0
Model               0
Kilometer           0
FuelType            0
Brand               0
NotRepaired         0
dtype: int64

Произведем кодирование категориальных данных с помощью LabelEncoder

In [26]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
cat_features = ['VehicleType','Gearbox','Model','FuelType','Brand','NotRepaired']
label_encode = LabelEncoder()
for column in cat_features:
    label_encode.fit(df[column].astype('str'))
    df[column] = label_encode.transform(df[column].astype('str'))

Разделим данные на обучающую, валидационную выборки. Определим признаки.

In [27]:
df_train, df_test = train_test_split(df, test_size=0.4, random_state=12345)
df_test, df_valid = train_test_split(df_test, test_size=0.5, random_state=12345)

features_valid = df_valid.drop(['Price'], axis=1)
target_valid = df_valid['Price']
features_train = df_train.drop(['Price'], axis=1)
target_train = df_train['Price']
features_test = df_test.drop(['Price'], axis=1)
target_test = df_test['Price']
features = df.drop('Price', axis=1)
target = df['Price']

Масштабируем данные с помощью функции.

In [28]:
def scale_to_standard(train, test, valid):
    scaler = StandardScaler()
    numeric = ['RegistrationYear','Power','Kilometer']   
    scaler.fit(train[numeric])
    train[numeric] = scaler.transform(train[numeric])
    valid[numeric]=scaler.transform(valid[numeric])
    test[numeric] = scaler.transform(test[numeric])
scale_to_standard(features_train, features_valid, features_test)

# Вывод

На данном шаге:

- Импортировали библиотеки.
- Открыли файл и изучили данные.
- Провели предобработку данных.
- Пропущенные значения заполнили.
- Аномальные значения удалили.
- Произвели кодирование категориальных данных 
- Разделили данные на обучающую, валидационную выборки. 
- Масштабировали данные.

## Обучение моделей

На данном этапе:
- обучим разные модели. 
- подберем гиперпараметры.

Подберем лучшие параметры для модели RandomForestRegressor с помощью функции GridSearchCV

In [29]:
rfr = RandomForestRegressor()
parametrs = { 'n_estimators': range (10, 51, 10),
             'max_depth': range (1,13, 2)}
grid = GridSearchCV(rfr, parametrs, scoring='neg_mean_squared_error')
grid.fit(features_train,target_train)
print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 11, 'n_estimators': 50}
-4084252.055419804


Подберем лучшие параметры для модели CatBoostRegressor с помощью функции GridSearchCV

In [30]:
cbr = CatBoostRegressor()
parametrs = {'depth': range (1,10, 2)}
grid = GridSearchCV(cbr, parametrs, scoring='neg_mean_squared_error')
grid.fit(features_train,target_train, cat_features=cat_features)
print(grid.best_params_)
print(grid.best_score_)

Learning rate set to 0.09022
0:	learn: 4443.2898998	total: 81.9ms	remaining: 1m 21s
1:	learn: 4337.4755917	total: 114ms	remaining: 56.8s
2:	learn: 4247.9045133	total: 135ms	remaining: 45s
3:	learn: 4167.2910526	total: 167ms	remaining: 41.6s
4:	learn: 4090.6923938	total: 186ms	remaining: 37.1s
5:	learn: 4023.2043713	total: 206ms	remaining: 34.1s
6:	learn: 3959.1796527	total: 225ms	remaining: 31.9s
7:	learn: 3902.7580587	total: 245ms	remaining: 30.3s
8:	learn: 3846.9006209	total: 263ms	remaining: 29s
9:	learn: 3798.1533961	total: 281ms	remaining: 27.8s
10:	learn: 3750.7853368	total: 301ms	remaining: 27.1s
11:	learn: 3709.1245713	total: 320ms	remaining: 26.3s
12:	learn: 3668.8076936	total: 339ms	remaining: 25.8s
13:	learn: 3631.4016378	total: 359ms	remaining: 25.3s
14:	learn: 3595.7143184	total: 383ms	remaining: 25.1s
15:	learn: 3564.0340842	total: 402ms	remaining: 24.7s
16:	learn: 3533.0833579	total: 421ms	remaining: 24.3s
17:	learn: 3503.6248261	total: 441ms	remaining: 24s
18:	learn: 34

Подберем лучшие параметры для модели LGBMRegressor с помощью функции GridSearchCV 

In [31]:
LGBM = LGBMRegressor()
parametrs = { 'n_estimators': range (10, 51, 10),
             'max_depth': range (1,13, 2)}
grid = GridSearchCV(LGBM, parametrs, scoring='neg_mean_squared_error')
grid.fit(features_train,target_train, categorical_feature=cat_features)
print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 11, 'n_estimators': 50}
-3477453.4548470764


Подберем лучшие параметры для модели XGBRegressor с помощью функции GridSearchCV

In [32]:
xgboost = XGBRegressor()
parametrs = {'max_depth': range (1,13,2)}
grid = GridSearchCV(xgboost, parametrs, scoring='neg_mean_squared_error')
grid.fit(features_train,target_train)
print(grid.best_params_)
print(grid.best_score_)

{'max_depth': 9}
-3142604.9739646767


Обучим модель RandomForestRegressor с подобранными параметрами на валидационной выборке.

In [33]:
%%time
model = RandomForestRegressor(n_estimators=40, 
    max_depth=11, random_state=12345)

model.fit(features_train, target_train) 
start = time.time()
predictions_valid = model.predict(features_valid) # обучите модель на обучающей выборке
end =  time.time()
RF_time = end - start 
print("Время обучения: ",RF_time)
print("RMSE на валидационной выборке: ", mean_squared_error(target_valid, predictions_valid)**0.5)# найдите значение метрики R2 на валидационной выборке

Время обучения:  0.2511632442474365
RMSE на валидационной выборке:  2041.6379652895926
CPU times: user 9.94 s, sys: 14.3 ms, total: 9.95 s
Wall time: 9.97 s


Обучим модель CatBoostRegressor с подобранными параметрами.

In [34]:
%%time
model = CatBoostRegressor(depth= 9, iterations=1000, random_state=12345)
model.fit(features_train, target_train,cat_features=cat_features, eval_set=(features_valid,target_valid)) 
start = time.time()
predictions_valid = model.predict(features_valid) 
end =  time.time()
cat_time = end - start 
print("Время обучения: ",cat_time)
print("RMSE на валидационной выборке: ", mean_squared_error(target_valid, predictions_valid)**0.5)

Learning rate set to 0.115628
0:	learn: 4199.7966955	test: 4196.1756820	best: 4196.1756820 (0)	total: 441ms	remaining: 7m 20s
1:	learn: 3888.5793326	test: 3888.7811421	best: 3888.7811421 (1)	total: 797ms	remaining: 6m 37s
2:	learn: 3620.8205764	test: 3624.2567617	best: 3624.2567617 (2)	total: 1.1s	remaining: 6m 6s
3:	learn: 3387.9187721	test: 3392.7745710	best: 3392.7745710 (3)	total: 1.49s	remaining: 6m 11s
4:	learn: 3189.7770834	test: 3197.1090337	best: 3197.1090337 (4)	total: 1.82s	remaining: 6m 2s
5:	learn: 3020.5322218	test: 3029.5816960	best: 3029.5816960 (5)	total: 2.18s	remaining: 6m 1s
6:	learn: 2878.1228561	test: 2889.8913733	best: 2889.8913733 (6)	total: 2.5s	remaining: 5m 55s
7:	learn: 2753.9942510	test: 2766.8347264	best: 2766.8347264 (7)	total: 2.94s	remaining: 6m 3s
8:	learn: 2648.2223924	test: 2662.3450469	best: 2662.3450469 (8)	total: 3.3s	remaining: 6m 3s
9:	learn: 2560.4080907	test: 2576.4702092	best: 2576.4702092 (9)	total: 3.67s	remaining: 6m 3s
10:	learn: 2480.943

Обучим модель LGBMRegressor с подобранными параметрами.

In [35]:
%%time
model = LGBMRegressor(max_depth = 11, n_estimators = 50, random_state=12345)
model.fit(features_train, target_train, categorical_feature=cat_features, verbose=100)  
start = time.time()
predictions_valid = model.predict(features_valid) 
end =  time.time()
LGBM_time = end - start 
print("Время обучения: ",LGBM_time)
print("RMSE на валидационной выборке: ", mean_squared_error(target_valid, predictions_valid)**0.5)

Время обучения:  0.32776737213134766
RMSE на валидационной выборке:  1873.7302164169976
CPU times: user 23.3 s, sys: 139 ms, total: 23.5 s
Wall time: 23.6 s


Обучим модель XGBRegressor с подобранными параметрами.

In [36]:
%%time
model = XGBRegressor(max_depth = 9, subsample= 1.0, random_state=12345)
model.fit(features_train, target_train, verbose=100)  
start = time.time()
predictions_valid = model.predict(features_valid) 
end =  time.time()
XGB_time = end - start 
print("Время обучения: ",XGB_time)
print("RMSE на валидационной выборке: ", mean_squared_error(target_valid, predictions_valid)**0.5)

Время обучения:  0.3825232982635498
RMSE на валидационной выборке:  1781.144873344275
CPU times: user 3min 16s, sys: 1.31 s, total: 3min 17s
Wall time: 3min 18s


Обучим модель RandomForestRegressor с подобранными параметрами на тестовой выборке.

In [37]:
%%time
model = RandomForestRegressor(n_estimators=50, 
    max_depth=11, random_state=12345)

model.fit(features_train, target_train)  
predictions_test = model.predict(features_test) 
print("Наилучшая модель")
print("RMSE на тестовой выборке: ", mean_squared_error(target_test, predictions_test)**0.5)# найдите значение метрики R2 на валидационной выборке

Наилучшая модель
RMSE на тестовой выборке:  2051.338263603141
CPU times: user 14.9 s, sys: 855 µs, total: 14.9 s
Wall time: 15.1 s


Обучим модель CatBoostRegressor с подобранными параметрами на тестовой выборке.

In [38]:
%%time
model= CatBoostRegressor(depth= 9, iterations= 100, random_state=12345)
model.fit(features_train, target_train, cat_features=cat_features, eval_set=(features_valid,target_valid)) 
predictions_test = model.predict(features_test) 
print("Наилучшая модель")
print("RMSE на тестовой выборке: ", mean_squared_error(target_test, predictions_test)**0.5)

Learning rate set to 0.471044
0:	learn: 3256.3939544	test: 3261.4007631	best: 3261.4007631 (0)	total: 131ms	remaining: 12.9s
1:	learn: 2649.3757528	test: 2663.2791908	best: 2663.2791908 (1)	total: 254ms	remaining: 12.5s
2:	learn: 2352.6916769	test: 2367.6049187	best: 2367.6049187 (2)	total: 386ms	remaining: 12.5s
3:	learn: 2197.0489226	test: 2217.9696344	best: 2217.9696344 (3)	total: 498ms	remaining: 12s
4:	learn: 2110.8060103	test: 2129.2959365	best: 2129.2959365 (4)	total: 611ms	remaining: 11.6s
5:	learn: 2077.6089815	test: 2098.7308887	best: 2098.7308887 (5)	total: 707ms	remaining: 11.1s
6:	learn: 2052.3309159	test: 2071.6012548	best: 2071.6012548 (6)	total: 803ms	remaining: 10.7s
7:	learn: 2028.5277400	test: 2050.6116791	best: 2050.6116791 (7)	total: 902ms	remaining: 10.4s
8:	learn: 2012.1033379	test: 2035.8508904	best: 2035.8508904 (8)	total: 997ms	remaining: 10.1s
9:	learn: 1994.4391564	test: 2022.8455003	best: 2022.8455003 (9)	total: 1.09s	remaining: 9.86s
10:	learn: 1981.641624

Обучим модель LGBMRegressor с подобранными параметрами на тестовой выборке.

In [39]:
%%time
model = LGBMRegressor(max_depth = 11, n_estimators = 50, random_state=12345)
model.fit(features_train, target_train, categorical_feature=cat_features, verbose=100)  
predictions_test = model.predict(features_test) 
print("Наилучшая модель")
print("RMSE на тестовой выборке: ", mean_squared_error(target_test, predictions_test)**0.5)

Наилучшая модель
RMSE на тестовой выборке:  1870.2445727060976
CPU times: user 1min 28s, sys: 483 ms, total: 1min 29s
Wall time: 1min 29s


Обучим модель XGBRegressor с подобранными параметрами на тестовой выборке.

In [40]:
%%time
model = XGBRegressor(max_depth = 9, subsample= 1.0, random_state=12345)
model.fit(features_train, target_train, verbose=100)  
predictions_test = model.predict(features_test) 
print("Наилучшая модель")
print("RMSE на тестовой выборке: ", mean_squared_error(target_test, predictions_test)**0.5)

Наилучшая модель
RMSE на тестовой выборке:  1787.8496801670258
CPU times: user 2min 8s, sys: 478 ms, total: 2min 9s
Wall time: 2min 10s


# Вывод

На данном этапе:

- обучили модели RandomForestRegressor, CatBoostRegressor, LGBMRegressor, XGBRegressor .
- подобрали гиперпараметры.

## Анализ моделей

Проанализируем скорость работы и качество моделей:

Для заказчика важны:
- качество предсказания;
- скорость предсказания;
- время обучения.

In [ ]:
<div class="alert alert-block alert-warning">
<b>Изменения:</b> Внесла 
</div>

In [42]:
results = {
    'Model' : ['RandomForestRegressor', 'CatBoostRegressor','LGBMRegressor', 'XGBRegressor'],
    'Valid RMSE' :pd.Series([2041.6, 1755.7, 1873.7,1781.1]),
    'Valid Time, sec': pd.Series([10.0, 312, 23.6, 198]),
    'Time Prediction': pd.Series([0.25, 1, 0.32, 0.36]),
    'Test RMSE' :pd.Series([2051.3, 1817.8, 1870.2,1787.8]),
    'Test Time, sec': pd.Series([15.1, 12.4, 89, 130])
    }
display(pd.DataFrame(results))

,Model,Valid RMSE,"Valid Time, sec",Time Prediction,Test RMSE,"Test Time, sec"
0,RandomForestRegressor,2041.6,10.0,0.25,2051.3,15.1
1,CatBoostRegressor,1755.7,312.0,1.00,1817.8,12.4
2,LGBMRegressor,1873.7,23.6,0.32,1870.2,89.0
3,XGBRegressor,1781.1,198.0,0.36,1787.8,130.0


# Вывод

- Модель CatBoostRegressor на валидационной выборке показала наилучшую метрику RMSE.
- Модель XGBRegressor обладает на тестовой выборке наилучшим показателем метрики RMSE, но при этом наихудшим результатом времени.
- Модель CatBoostRegressor обладает на тестовой выборке вторым показателем метрики RMSE, но при этом лучший результат по времени.
- Время предсказания у модели CatBoostRegressor дольше всех остальных моделей. 